# Synthetic Test Dataset Creation for RAG Evaluation

This notebook generates a synthetic QA dataset used for evaluating retrieval quality in the RAG system.

Main steps:
- Load repository files
- Split documents into chunks
- Generate summaries
- Create evaluation questions
- Build final QA test dataset

## Imports and Environment Setup

Load required libraries, API configuration, and environment variables.

In [ ]:
# imports
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError
from litellm import completion
import json
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [2]:
MODEL = "gpt-4.1-mini"
openai = OpenAI()
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

ROOT = Path("..")
REPOS_DIR = ROOT / "repos"
DATA_DIR = ROOT / "data"

OpenAI API Key exists and begins sk-proj-


In [3]:
def load_ipynb(file_path):
    texts = []

    with open(file_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "markdown":
            texts.append("MARKDOWN:\n" + "".join(cell.get("source", [])))
        elif cell.get("cell_type") == "code":
            texts.append("CODE:\n" + "".join(cell.get("source", [])))

    return "\n".join(texts)

## Project Metadata Extraction
Utilities for detecting project names and organizing repository structure.

In [4]:
def extract_project_name(path: Path) -> str:
    parts = path.parts

    if "repos" in parts:
        idx = parts.index("repos")
        if idx + 1 < len(parts):
            return parts[idx + 1]

    return "UNKNOWN_PROJECT"

## Repository Loading

Load repository files and convert notebooks and markdown files into text documents.

In [5]:
def load_files():
    documents = []

    for file_path in REPOS_DIR.rglob("*"):

        if ".git" in file_path.parts:
            continue

        if file_path.suffix not in [".md", ".ipynb"]:
            continue

        print("Loading:", file_path)

        try:
            if file_path.suffix == ".ipynb":
                content = load_ipynb(file_path)
            else:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()

            project_name = extract_project_name(file_path)

            documents.append(
                Document(
                    page_content=content,
                    metadata={
                        "source": str(file_path),
                        "project": project_name
                    }
                )
            )

        except Exception as e:
            print("ERROR:", file_path, e)

    print(f"Loaded {len(documents)} documents")
    return documents

In [6]:
documents = load_files()

Loading: ..\repos\Career-chat-rag\career_chat_1.ipynb
Loading: ..\repos\Career-chat-rag\career_chat_2.ipynb
Loading: ..\repos\Career-chat-rag\creator_test_data.ipynb
Loading: ..\repos\Career-chat-rag\prepare_github_data.ipynb
Loading: ..\repos\Career-chat-rag\README.md
Loading: ..\repos\Car_brand_classification_CNN\Car_brand_classification_CNN.ipynb
Loading: ..\repos\Car_brand_classification_CNN\README.md
Loading: ..\repos\Cracow_Real_Estate_Price_Prediction_2025\Flats_feature_engineering.ipynb
Loading: ..\repos\Cracow_Real_Estate_Price_Prediction_2025\Flats_scraping.ipynb
Loading: ..\repos\Cracow_Real_Estate_Price_Prediction_2025\Models_results.ipynb
Loading: ..\repos\Cracow_Real_Estate_Price_Prediction_2025\README.md
Loading: ..\repos\LIDAR_point_cloud_classification_ANN\Lidar_point_cloud_classification.ipynb
Loading: ..\repos\LIDAR_point_cloud_classification_ANN\README.md
Loading: ..\repos\NLP-Classification_of_tweets_about_airlines_LSTM\Airlines_Sentiments_Analysis_LSTM.ipynb
Loadi

## Dataset Statistics

Analyze repository size and document distribution before chunk generation.

In [7]:
total_chars = sum(len(doc.page_content) for doc in documents)
print("Total characters:", total_chars)

chars_list= []
for doc in documents:
    chars_list.append(len(doc.page_content))
print("Characters in files:",chars_list)


Total characters: 186062
Characters in files: [18904, 29981, 16402, 1255, 5722, 16646, 738, 43359, 7645, 18526, 1032, 11684, 1266, 12087, 815]


## Chunking Strategy

Split repository content into overlapping chunks for evaluation preparation.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 250 chunks
First chunk:

page_content='MARKDOWN:
# AI Portfolio Assistant – Baseline Version

This notebook contains the initial implementation of an AI portfolio assistant based on Retrieval-Augmented Generation (RAG).

The goal of this version was to build and validate the complete RAG pipeline before introducing optimization techniques.

Main components:' metadata={'source': '..\\repos\\Career-chat-rag\\career_chat_1.ipynb', 'project': 'Career-chat-rag'}


## Chunk Summarization

Generate concise summaries for chunks to support creation of higher-quality evaluation questions.

In [9]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [10]:
class Chunk(BaseModel):
    headline: str
    summary: str
    original_text: str

    def as_result(self, document):
        return Result(
            page_content=f"{self.headline}\n\n{self.summary}\n\n{self.original_text}",
            metadata=document.metadata
        )

In [11]:
def make_prompt(chunk_text):
    return f"""
You are an expert AI assistant summarizing technical code and markdown.

Your task:
- Create a short headline
- Create a short summary

IMPORTANT RULES:
- Use ONLY the provided text
- Keep summary concise (2-3 sentences)
- Return ONLY valid JSON

{{
  "headline": "short title",
  "summary": "short summary"
}}

TEXT:
{chunk_text}
"""

In [12]:
def process_chunk(chunk, retries=3):
    for i in range(retries):
        try:
            prompt = make_prompt(chunk.page_content)

            response = completion(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0
            )

            content = response.choices[0].message.content

            data = json.loads(content)

            parsed = Chunk(
                headline=data["headline"],
                summary=data["summary"],
                original_text=chunk.page_content
            )

            return parsed.as_result(chunk)

        except Exception as e:
            print(f"Retry {i+1}: {e}")

    return Result(
        page_content=f"""TITLE: FALLBACK

SUMMARY: Failed to parse structured chunk, using raw content

CONTENT:
{chunk.page_content[:1000]}""",
        metadata=chunk.metadata
    )

In [13]:
def process_all_chunks(chunks):
    return [process_chunk(c) for c in chunks]

In [14]:
chunks_result = [process_chunk(chunk) for chunk in chunks]

## Test Schema Definition

Define the structure and validation rules for generated QA examples.

In [15]:
class QA(BaseModel):
    question: str
    reference_answer: str
    category: str
    keywords: list[str]


class QAResponse(BaseModel):
    qa_pairs: list[QA]

In [16]:
def truncate_context(text, max_chars=10000):
    return text[:max_chars]

In [17]:
def validate_qa(qa_list, mode="chunk"):
    valid = []

    for qa in qa_list:
        if not qa.question or not qa.reference_answer:
            continue

        if not isinstance(qa.keywords, list):
            continue

        if mode == "global":
            allowed = ["cross_context", "comparative_reasoning", "local_reasoning"]
            if qa.category not in allowed:
                continue

        valid.append(qa)

    return valid

In [18]:
def parse_qa_response(text):
    try:
        data = json.loads(text)
        return QAResponse.model_validate(data).qa_pairs

    except ValidationError as e:
        print("❌ Pydantic error:", e)
        return []

    except Exception as e:
        print("❌ JSON error:", e)
        return []

In [19]:
def build_project_prompt(context, project_name, n):
    return f"""
You are generating a high-quality QA dataset for RAG evaluation.

========================
PROJECT
========================
{project_name}

========================
HARD RULES
========================
- Every question MUST start with:
  "In {project_name}, ..."
- Use ONLY the provided context
- Do NOT hallucinate
- Generate EXACTLY {n} QA pairs
- Each question must refer ONLY to this project
- Answers must be factual and grounded in context
- Avoid generic questions like:
  - "What is the purpose..."
  - "What is the role..."
- Focus on:
  - concrete values
  - comparisons
  - specific implementation details

========================
OUTPUT FORMAT (STRICT JSON)
========================
{{
  "qa_pairs": [
    {{
      "question": "...",
      "reference_answer": "...",
      "category": "atomic_fact | local_reasoning | cross_context | comparative_reasoning",
      "keywords": ["..."]
    }}
  ]
}}

========================
CONTEXT
========================
{context}
"""

## QA Test Generation

Generate synthetic question–answer pairs used later to benchmark retrieval quality.

In [20]:
def generate_qa_chunk(chunks, n=1):
    all_qa = []

    for chunk in chunks:
        project_name = chunk.metadata.get("project", "UNKNOWN_PROJECT")

        context = truncate_context(chunk.page_content)

        prompt = build_project_prompt(
            context=context,
            project_name=project_name,
            n=n
        )

        try:
            response = completion(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                response_format={"type": "json_object"}
            )

            qa = parse_qa_response(response.choices[0].message.content)
            qa = validate_qa(qa, mode="chunk")

            all_qa.extend(qa)

        except Exception as e:
            print("Chunk QA error:", e)

    return all_qa

In [21]:
qa_chunk = generate_qa_chunk(chunks_result)

In [22]:
len(qa_chunk)

250

## Project-Level Question Generation

Create broader questions that require understanding information across a project.

In [23]:
def group_by_project(results):
    grouped = {}

    for r in results:
        project = r.metadata.get("project", "UNKNOWN_PROJECT")

        if project not in grouped:
            grouped[project] = []

        grouped[project].append(r)

    return grouped

In [24]:
def build_project_context(results):
    summaries = []

    for r in results:
        parts = r.page_content.split("\n\n")

        if len(parts) >= 2:
            summaries.append(parts[1]) 

    return "\n".join(summaries)

In [25]:
def generate_qa_project(results, n_per_project=50):
    grouped = group_by_project(results)

    all_qa = []

    for project_name, items in grouped.items():

        context = build_project_context(items)
        context = truncate_context(context)

        prompt = build_project_prompt(
            context=context,
            project_name=project_name,
            n=n_per_project
        )

        try:
            response = completion(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                response_format={"type": "json_object"}
            )

            qa = parse_qa_response(response.choices[0].message.content)
            qa = validate_qa(qa, mode="global")

            all_qa.extend(qa)

        except Exception as e:
            print(f"Global QA error ({project_name}):", e)

    return all_qa

In [26]:
qa_project = generate_qa_project(chunks_result)

In [27]:
len(qa_project)

34

## Cross-Project Evaluation Cases

Generate examples that combine information from multiple repositories.

In [28]:
def build_cross_context(results):
    projects = group_by_project(results)

    blocks = []

    for project, items in projects.items():
        summaries = build_project_context(items)
        blocks.append(f"PROJECT: {project}\n{summaries}")

    return "\n\n".join(blocks)

In [29]:
def parse_qa_response_safe(text):
    try:
        data = json.loads(text)
        qa_pairs = data.get("qa_pairs", [])

        fixed = []

        for qa in qa_pairs:
            if not isinstance(qa, dict):
                continue

            # 🔥 fallbacki
            fixed.append(QA(
                question=qa.get("question", ""),
                reference_answer=qa.get("reference_answer", qa.get("answer", "")),
                category=qa.get("category", "cross_context"),
                keywords=qa.get("keywords", [])
            ))

        return fixed

    except Exception as e:
        print("❌ parse error:", e)
        return []

In [30]:
def enforce_cross_project_rules(qa_list, project_names):
    valid = []

    for qa in qa_list:
        q = qa.question.lower()

        if not q.startswith("across projects"):
            continue

        matches = [p for p in project_names if p.lower() in q]

        if len(matches) < 2:
            continue

        valid.append(qa)

    return valid

In [31]:
def build_cross_project_prompt(context, n, projects_str):
    return f"""
You are generating a structured QA dataset for RAG evaluation.

========================
AVAILABLE PROJECTS
========================
{projects_str}

========================
HARD REQUIREMENTS
========================
- Generate EXACTLY {n} QA pairs. If a QA is too generic, refine it instead of skipping.
- Each QA pair MUST contain ALL fields:
  - question (string)
  - reference_answer (string)
  - category (string)
  - keywords (list of strings)

- DO NOT omit any field
- DO NOT rename fields
- DO NOT return partial objects

========================
QUESTION RULES
========================
- Each question MUST explicitly mention at least 2 different projects
- Questions referring to only one project are INVALID
- Each question MUST start with:
  "Across projects, ..."
- Avoid generic comparisons
- Include specific methods, metrics, or transformations from the context
- Prefer questions that require retrieving concrete details (e.g. metrics, techniques, features)
- Generate cross-project questions using different reasoning types:
  - comparison of techniques
  - failure scenarios
  - suitability analysis
  - trade-off reasoning
  - edge cases
  - robustness under noise
  - what-if scenarios
- Avoid repeating the same "how do X differ between A and B" structure.

========================
CATEGORIES (STRICT)
========================
Use ONLY:
- cross_context
- comparative_reasoning

========================
OUTPUT FORMAT (STRICT JSON)
========================
{{
  "qa_pairs": [
    {{
      "question": "string",
      "reference_answer": "string",
      "category": "cross_context | comparative_reasoning",
      "keywords": ["string"]
    }}
  ]
}}

Return ONLY JSON. No explanations.

========================
CONTEXT
========================
{context}
"""

In [32]:
def generate_cross_project_qa(results, n=10, retries=3):
    grouped = group_by_project(results)

    projects = list(grouped.keys())
    projects_str = "\n".join([f"- {p}" for p in projects])

    context = truncate_context(build_cross_context(results))

    prompt = build_cross_project_prompt(
        context=context,
        n=n,
        projects_str=projects_str
    )

    for i in range(retries):
        try:
            response = completion(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                response_format={"type": "json_object"}
            )

            content = response.choices[0].message.content

            qa = parse_qa_response_safe(content)
            qa = enforce_cross_project_rules(qa, projects)
            qa = validate_qa(qa, mode="global")

            if len(qa) >= max(1, n // 2):
                return qa

        except Exception as e:
            print(f"Retry {i+1} error:", e)

    return []

In [33]:
qa_cross_project = generate_cross_project_qa(chunks_result)

In [34]:
len(qa_cross_project)

10

In [35]:
all_qa = qa_chunk  + qa_project + qa_cross_project

In [36]:
len(all_qa)

294

## Dataset Cleaning and Validation

Remove duplicates and filter low-quality or ambiguous samples.

In [37]:
def deduplicate_qa(qa_list):
    seen = set()
    unique = []

    for qa in qa_list:
        key = qa.question.strip().lower()

        if key not in seen:
            seen.add(key)
            unique.append(qa)

    return unique

In [38]:
def is_trivial_too_generic(q):
    bad_patterns = [
        "what is",
        "which is",
        "what are",
        "what type",
        "what parameter",
        "what metric",
        "what is the purpose",
        "what is the role",
        "what is the significance",
        "what is the main goal",
        "what is used",
        "what does the code do"
    ]
    
    q_lower = q.question.lower()
    return any(p in q_lower for p in bad_patterns)

In [39]:
all_qa = deduplicate_qa(all_qa)
all_qa = [q for q in all_qa if not is_trivial_too_generic(q)]

In [40]:
len(all_qa)

215

In [41]:
all_qa[:10]

[QA(question='In Career-chat-rag, what was the primary goal of the baseline AI portfolio assistant implementation?', reference_answer='The primary goal of the baseline AI portfolio assistant implementation in Career-chat-rag was to build and validate the complete Retrieval-Augmented Generation (RAG) pipeline before introducing optimization techniques.', category='atomic_fact', keywords=['baseline implementation', 'goal', 'RAG pipeline', 'validation']),
 QA(question='In Career-chat-rag, which embedding model is configured for document processing and retrieval?', reference_answer='The embedding model configured is "text-embedding-3-small".', category='atomic_fact', keywords=['embedding model', 'configuration', 'document processing', 'retrieval']),
 QA(question='In Career-chat-rag, what does the function extract_project_name return when given an empty path string?', reference_answer='The function extract_project_name returns the string "UNKNOWN_PROJECT" when given an empty path string.', 

## Export Test Dataset

Save the generated evaluation dataset for later RAG benchmarking.

In [42]:
with open(DATA_DIR / "tests.jsonl", "w", encoding="utf-8") as f:
    for qa in all_qa:
        f.write(json.dumps(qa.model_dump(), ensure_ascii=False) + "\n")

# Summary

This notebook generates a synthetic evaluation dataset used to benchmark the final RAG pipeline.

The generated QA pairs are later used to evaluate:
- retrieval relevance,
- context coverage,
- cross-document reasoning.